In [3]:
# 1. Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import modul Machine Learning
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

In [4]:
df_train = pd.read_excel('../data/kelulusan_train.xls')
df_test = pd.read_excel('../data/kelulusan_test.xls')

In [5]:
df_train

,NAMA,JENIS KELAMIN,STATUS MAHASISWA,UMUR,STATUS NIKAH,IPS 1,IPS 2,IPS 3,IPS 4,IPS 5,IPS 6,IPS 7,IPS 8,IPK,STATUS KELULUSAN
0,ANIK WIDAYANTI,PEREMPUAN,BEKERJA,28,BELUM MENIKAH,2.76,2.80,3.20,3.17,2.98,3.00,3.03,0.0,3.07,TERLAMBAT
1,DWI HESTYNA PRIHASTANTY,PEREMPUAN,MAHASISWA,32,BELUM MENIKAH,3.00,3.30,3.14,3.14,2.84,3.13,3.25,0.0,3.17,TERLAMBAT
2,MURYA ARIEF BASUKI,PEREMPUAN,BEKERJA,29,BELUM MENIKAH,3.50,3.30,3.70,3.29,3.53,3.72,3.73,0.0,3.54,TERLAMBAT
3,NANIK SUSANTI,PEREMPUAN,MAHASISWA,27,BELUM MENIKAH,3.17,3.41,3.61,3.36,3.48,3.63,3.46,0.0,3.41,TERLAMBAT
4,RIFKA ISTIQFARINA,PEREMPUAN,BEKERJA,29,BELUM MENIKAH,2.90,2.89,3.30,2.85,2.98,3.00,3.08,0.0,3.09,TERLAMBAT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374,ARY JULI SETIYANTO,LAKI - LAKI,MAHASISWA,23,BELUM MENIKAH,1.98,2.50,2.14,2.77,2.61,2.93,2.82,2.5,0.99,TEPAT
375,RINA ZAHROTUL UMAMI,PEREMPUAN,BEKERJA,23,BELUM MENIKAH,2.74,2.75,2.55,3.00,2.98,2.80,3.14,3.0,2.97,TEPAT
376,TULISA WAHYUHADI KRISNATAMI,PEREMPUAN,MAHASISWA,23,BELUM MENIKAH,2.74,2.75,2.55,3.00,2.98,2.80,3.14,3.0,3.03,TEPAT
377,NI'MATUL JANNAH,PEREMPUAN,MAHASISWA,23,BELUM MENIKAH,3.02,2.94,3.25,2.87,3.00,2.94,3.09,3.0,3.16,TEPAT


In [6]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 379 entries, 0 to 378
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   NAMA              379 non-null    object 
 1   JENIS KELAMIN     379 non-null    object 
 2   STATUS MAHASISWA  379 non-null    object 
 3   UMUR              379 non-null    int64  
 4   STATUS NIKAH      379 non-null    object 
 5   IPS 1             379 non-null    float64
 6   IPS 2             379 non-null    float64
 7   IPS 3             379 non-null    float64
 8   IPS 4             379 non-null    float64
 9   IPS 5             379 non-null    float64
 10  IPS 6             379 non-null    float64
 11  IPS 7             379 non-null    float64
 12  IPS 8             372 non-null    float64
 13  IPK               376 non-null    float64
 14  STATUS KELULUSAN  379 non-null    object 
dtypes: float64(9), int64(1), object(5)
memory usage: 44.5+ KB


In [7]:
# 3. Data Cleaning

# Pemeriksaan nilai unik kolom kategorikal (untuk konfirmasi)
print("\n--- Nilai Unik Kolom Kategorikal ---")
print("JENIS KELAMIN:", df_train['JENIS KELAMIN'].unique())
print("STATUS MAHASISWA:", df_train['STATUS MAHASISWA'].unique())
print("STATUS NIKAH:", df_train['STATUS NIKAH'].unique())
print("STATUS KELULUSAN:", df_train['STATUS KELULUSAN'].unique())

# Menghapus kolom 'STATUS NIKAH' (karena hanya memiliki satu nilai unik: 'BELUM MENIKAH')
df_train.drop(columns=['STATUS NIKAH'], inplace=True)
df_test.drop(columns=['STATUS NIKAH'], inplace=True)
print("\nKolom 'STATUS NIKAH' telah dihapus.")

# Mengubah Nilai Kategorikal menjadi Numerik (Encoding)
replacements = {
    'JENIS KELAMIN': {'LAKI-LAKI': 1, 'PEREMPUAN': 0},
    'STATUS MAHASISWA': {'MAHASISWA': 0, 'BEKERJA': 1},
    'STATUS KELULUSAN': {'TERLAMBAT': 1, 'TEPAT': 0}
}

df_train = df_train.replace(replacements, inplace=False)
df_test = df_test.replace(replacements, inplace=False)

print("\n--- Data Training Setelah Encoding ---")
print(df_train.head())


--- Nilai Unik Kolom Kategorikal ---
JENIS KELAMIN: ['PEREMPUAN' 'LAKI - LAKI']
STATUS MAHASISWA: ['BEKERJA' 'MAHASISWA']
STATUS NIKAH: ['BELUM MENIKAH' 'MENIKAH']
STATUS KELULUSAN: ['TERLAMBAT' 'TEPAT']

Kolom 'STATUS NIKAH' telah dihapus.

--- Data Training Setelah Encoding ---
                      NAMA JENIS KELAMIN  STATUS MAHASISWA  UMUR  IPS 1  \
0           ANIK WIDAYANTI             0                 1    28   2.76   
1  DWI HESTYNA PRIHASTANTY             0                 0    32   3.00   
2       MURYA ARIEF BASUKI             0                 1    29   3.50   
3            NANIK SUSANTI             0                 0    27   3.17   
4        RIFKA ISTIQFARINA             0                 1    29   2.90   

   IPS 2  IPS 3  IPS 4  IPS 5  IPS 6  IPS 7  IPS 8  IPK   STATUS KELULUSAN  
0   2.80   3.20   3.17   2.98   3.00   3.03    0.0  3.07                 1  
1   3.30   3.14   3.14   2.84   3.13   3.25    0.0  3.17                 1  
2   3.30   3.70   3.29   3.53   3.72

C:\Users\kenas\AppData\Local\Temp\ipykernel_21524\2922530394.py:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_train = df_train.replace(replacements, inplace=False)
C:\Users\kenas\AppData\Local\Temp\ipykernel_21524\2922530394.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_test = df_test.replace(replacements, inplace=False)


In [8]:
# Cek Missing Value
print("\n--- Missing Value Sebelum Dropna ---")
print("df_train:\n", df_train.isnull().sum().loc[['IPS 8', 'IPK']])
print("df_test:\n", df_test.isnull().sum().loc[['IPS 8', 'IPK']])

# Menghapus baris dengan Missing Value pada kolom 'IPS 8' dan 'IPK'
df_train = df_train.dropna(subset=['IPS 8', 'IPK'])
df_test = df_test.dropna(subset=['IPS 8', 'IPK'])

# Cek kembali Missing Value
print("\n--- Missing Value Setelah Dropna ---")
print("df_train:\n", df_train.isnull().sum().loc[['IPS 8', 'IPK']])
print("df_test:\n", df_test.isnull().sum().loc[['IPS 8', 'IPK']])


--- Missing Value Sebelum Dropna ---


KeyError: "['IPK'] not in index"